# Time-domain Maxwell MHM in two and three dimensions

Tangential hybridization of central DG operators, physical material masses and leapfrog evolution. Electric and magnetic fields are compared at their different staggered times. Broken curl errors do not establish global H(curl) conformity.

The current main equations and their physical checks execute from source. The full acquisition and its field galleries run with `--study`; inspect `--study --plan` for locked commands and resource requirements. Exact original external comparisons require `--historical` and their attributed payloads.


In [ ]:
from pathlib import Path
import sys

# Locate the checkout from the repository root or any notebook subdirectory.
ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file()
     and (path / "src" / "pymhm").is_dir()
     and (path / "notebooks").is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError("Run this notebook from the PyMHM checkout.")
root = ROOT
for directory in (ROOT, ROOT / "examples"):
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))

import os
HISTORICAL_REPLAY = os.environ.get("PYMHM_NOTEBOOK_HISTORICAL", "0") == "1"
STUDY_REPLAY = os.environ.get("PYMHM_NOTEBOOK_STUDY", "0") == "1"


In [ ]:
from pathlib import Path
import json
import sys
import numpy as np
from IPython.display import Image, display
from threadpoolctl import threadpool_limits

sys.path.insert(0, str(ROOT))


## User-defined time-discrete equations

The primary loop uses the shared free equation builders displayed below.
`LocalEquations` declares each local midpoint field and its signed trace maps;
`Equation` supplies the exterior impedance balance. The magnetic update has
zero trace coordinates and uses the same global assembly/reconstruction core.
With $C=-\mathrm{curl}$, the electric stage and global face equation are

$$
\begin{aligned}
 M_e w+\tfrac{\tau}{2}Q\lambda
 &=M_e E_{\mathrm{old}}+\tfrac{\tau}{2}(F-C^\top H),\\
 -\sum_K Q_K^\top w_K+Z\lambda&=-b,\\
 E_{\mathrm{new}}&=2w-E_{\mathrm{old}}.
\end{aligned}
$$

The initialization takes a half electric kick after the physical mass and PEC
projections. Each later step first solves

$$
 M_h H_{\mathrm{new}}=M_h H_{\mathrm{old}}+\Delta t\,C E.
$$

The helper accepts explicit material/boundary callbacks, face partitions and
execution settings. A physical solver name does not select these equations.

In [ ]:
import inspect
from IPython.display import Code
from examples.tutorial_maxwell_equations import (
    prepare, initialize, advance, l2_errors, compare_state,
    electric_equations, magnetic_equations, electric_global,
)

# These are the exact source forms consumed by the generic core below.
for builder in (electric_equations, magnetic_equations, electric_global):
    display(Code(inspect.getsource(builder), language="python"))

In [ ]:
from pymhm.meshes.tetrahedron import TetraMesh
from examples.maxwell_data import CavityMode

mode = CavityMode(3)
mesh = TetraMesh.unit_cube()
with threadpool_limits(1):
    operator = prepare(
        mesh, time_step=0.001, degree=3, local_refinement=1,
        absorbing=None, quadrature_order=6,
    )
    first = initialize(operator, mode.electric_shape, 0.0)
    trajectory = [first]
    solution = first
    for _ in range(5):
        solution = advance(operator, solution)
        trajectory.append(solution)
        assert abs(solution.energy_balance_residual) < 2e-13
    drift = abs(solution.energy / first.energy - 1)
assert drift < 2e-13
{
    "electric_time": solution.electric_time,
    "magnetic_time": solution.magnetic_time,
    "relative_cross_time_energy_drift": drift,
}

## Direct original-matrix comparison

The independent loop owns the primary time updates. A separate solve of the
original A/B/C/D rows uses the identical spaces, data and five steps without
Schur elimination. The comparison checks coefficients, staggered times,
field norms and energy. It is a comparison of the same discrete equations,
not an independently discretized physical reference. The later records
retain their acquisition provenance.


In [ ]:
with threadpool_limits(1):
    references = [initialize(operator, mode.electric_shape, 0.0, original=True)]
    for _ in range(5):
        reference = advance(operator, references[-1], original=True)
        assert abs(reference.energy_balance_residual) < 2e-13
        references.append(reference)
    reference_drift = abs(references[-1].energy / references[0].energy - 1)
assert reference_drift < 2e-13
for own, reference in zip(trajectory, references, strict=True):
    differences = compare_state(own, reference)
    assert max(differences.values()) < 2e-11
    errors = l2_errors(
        operator, own,
        lambda points: mode.electric(own.electric_time, points),
        lambda points: mode.magnetic(own.magnetic_time, points), order=6,
    )
    comparator_errors = reference.l2_errors(
        lambda points: mode.electric(reference.electric_time, points),
        lambda points: mode.magnetic(reference.magnetic_time, points), order=6,
    )
    np.testing.assert_allclose(errors, comparator_errors, atol=2e-15)
print({"primary_energy_drift": drift, "original_matrix_energy_drift": reference_drift})

## Five-level spatial and temporal studies

The TM cavity uses the analytical data of Lanteri et al. (2018), one local P(ell+2) element and polynomial face degree ell. The three-dimensional mode is an original full-vector verification. The temporal baseline is the exact exponential of the same constrained spatial ODE.

In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    record = json.loads((ROOT / "examples/results/maxwell/comparison.json").read_text())
    assert record["source_changed_during_run"] is False
    [(r["dimension"], r["resolution"], r["trace_degree"], r["max_in_time"]["combined_l2"],
      r["max_in_time"]["combined_hcurl"], r["modified_energy_relative_drift"])
     for r in record["rows"] if r["study"] == "cavity"]

In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    for name in ("convergence", "time-energy", "cavity-2d-ell2-n8-fields", "cavity-3d-ell1-n4-electric", "cavity-3d-ell1-n4-magnetic"):
        display(Image(filename=str(ROOT / f"docs/figures/maxwell/{name}.png")))

Every spatial panel preserves signs and incident DG values and overlays the actual macro mesh or its physical section. Energy conservation, algebraic residuals and field accuracy are distinct checks. The quadrilateral heterogeneous nano-waveguide is a separate benchmark.